# Reco-Only Particle Identification (PID) Study

This notebook implements a complete particle-ID study using **reconstructed-only features** (length, hit counts, energy loss, straightness, ranking, and DCA to reconstructed vertex) to classify particle types. 
Truth information (`Truth_Info`) is used **only** to assign labels for training and validation.

In [ ]:
import uproot
import awkward as ak
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.dpi": 100,
})

print("Imports successful! ✓")

## 1. Load ROOT Trees and Extract Features

In [ ]:
file_path = Path("../Cut2.root")
if not file_path.exists():
    raise FileNotFoundError(f"Could not find {file_path.resolve()}")

f = uproot.open(file_path)
t_rc = f["Reco_Tree;2"]
t_tr = f["Truth_Info;2"]

rc_arr = t_rc.arrays(["TrackHitPos", "TrackHitEnergies", "nTracks"])
tr_arr = t_tr.arrays([
    "TrueVtxX", "TrueVtxY", "TrueVtxZ", 
    "TrueVtxID", "RecoTrackPrimaryParticleVtxId", 
    "RecoTrackPrimaryParticlePDG"
])
print(f"Loaded {len(rc_arr)} events.")

## 2. Feature Compilation Loop

We calculate 3D track properties and reco vertices. We also compute event-level properties such as length rank and Z-penetration rank.

In [ ]:
def fit_3d_line(hits):
    mean = np.mean(hits, axis=0)
    centered = hits - mean
    cov = np.cov(centered, rowvar=False)
    if cov.ndim < 2:
        return mean, np.array([0.0, 0.0, 1.0]), 0.0
    eigenvalues, eigenvectors = np.linalg.eigh(cov)
    direction = eigenvectors[:, np.argmax(eigenvalues)]
    
    diff = centered
    proj = diff - np.outer(diff @ direction, direction)
    residuals = np.sum(proj**2, axis=1)
    mean_residual = np.mean(residuals)
    return mean, direction, mean_residual

def get_dca_and_angle(vtx, p, d, start_hit):
    v_to_p = vtx - p
    dca = np.linalg.norm(v_to_p - (v_to_p @ d) * d)
    v_to_start = start_hit - vtx
    norm = np.linalg.norm(v_to_start)
    if norm > 0:
        cos_angle = (v_to_start @ d) / norm
        angle = np.arccos(np.clip(cos_angle, -1.0, 1.0)) * 180.0 / np.pi
    else:
        angle = 0.0
    return dca, angle

In [ ]:
track_records = []

for event_id in range(len(rc_arr)):
    n_tracks = rc_arr["nTracks"][event_id]
    if n_tracks < 1:
        continue
        
    track_pdgs = tr_arr["RecoTrackPrimaryParticlePDG"][event_id].tolist()
    
    # Fit tracks
    lines = []
    track_hits = []
    track_energies = []
    
    for track_idx in range(n_tracks):
        hits = ak.to_numpy(rc_arr["TrackHitPos"][event_id][track_idx])
        energies = ak.to_numpy(rc_arr["TrackHitEnergies"][event_id][track_idx])
        mask = hits[:, 0] > -9e8
        actual_hits = hits[mask]
        actual_energies = energies[mask]
        
        if len(actual_hits) >= 2:
            p, d, resid = fit_3d_line(actual_hits)
            lines.append((p, d, resid))
            track_hits.append(actual_hits)
            track_energies.append(actual_energies)
            
    if len(lines) == 0:
        continue
        
    # Reconstruct vertex (reco-only)
    if len(lines) == 1:
        p, d, resid = lines[0]
        actual_hits = track_hits[0]
        reco_vtx = actual_hits[np.argmin(actual_hits[:, 2])]
    else:
        A = np.zeros((3, 3))
        b = np.zeros(3)
        for p, d, resid in lines:
            proj = np.eye(3) - np.outer(d, d)
            A += proj
            b += proj @ p
        try:
            reco_vtx = np.linalg.solve(A, b)
        except np.linalg.LinAlgError:
            reco_vtx = np.mean([p for p, d, resid in lines], axis=0)
(

In [ ]:
    # Calculate track features
    event_tracks = []
    for track_idx, (p, d, resid) in enumerate(lines):
        actual_hits = track_hits[track_idx]
        actual_energies = track_energies[track_idx]
        
        start_hit = actual_hits[np.argmin(actual_hits[:, 2])]
        end_hit = actual_hits[np.argmax(actual_hits[:, 2])]
        length = np.linalg.norm(end_hit - start_hit)
        nhits = len(actual_hits)
        
        dca, angle = get_dca_and_angle(reco_vtx, p, d, start_hit)
        dist_to_vtx = np.linalg.norm(start_hit - reco_vtx)
        z_depth = np.max(actual_hits[:, 2]) - np.min(actual_hits[:, 2])
        
        total_energy = np.sum(actual_energies)
        dedx = total_energy / length if length > 0 else 0.0
        
        pdg = track_pdgs[track_idx] if track_idx < len(track_pdgs) else 0
        abs_pdg = abs(pdg)
        
        if abs_pdg == 13:
            label = "muon"
        elif abs_pdg == 211:
            label = "pion"
        elif pdg == 2212:
            label = "proton"
        elif abs_pdg == 11:
            label = "electron"
        else:
            label = "other"
            
        event_tracks.append({
            "pdg": pdg,
            "label": label,
            "length": length,
            "nhits": nhits,
            "dca": dca,
            "angle": angle,
            "dist_to_vtx": dist_to_vtx,
            "z_depth": z_depth,
            "start_z": start_hit[2],
            "end_z": end_hit[2],
            "straightness": resid,
            "total_energy": total_energy,
            "dedx": dedx
        })
        
    # Sort and rank tracks by length
    sorted_by_len = sorted(range(len(event_tracks)), key=lambda k: event_tracks[k]["length"], reverse=True)
    for rank, idx in enumerate(sorted_by_len):
        event_tracks[idx]["length_rank"] = rank + 1
        event_tracks[idx]["is_longest"] = 1 if rank == 0 else 0
        
    # Sort and rank tracks by z_depth
    sorted_by_z = sorted(range(len(event_tracks)), key=lambda k: event_tracks[k]["z_depth"], reverse=True)
    for rank, idx in enumerate(sorted_by_z):
        event_tracks[idx]["z_depth_rank"] = rank + 1
        
    track_records.extend(event_tracks)

df = pd.DataFrame(track_records)
print(f"Dataset compiled: {len(df)} tracks.")

## 3. Compare Feature Distributions by Particle Class

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
features_to_plot = ["length", "nhits", "z_depth", "dedx"]
titles = ["Track Length (mm)", "Number of Hits", "Z-Penetration Depth (mm)", "Energy Loss (dE/dx)"]

for i, feat in enumerate(features_to_plot):
    ax = axes[i // 2][i % 2]
    for lbl in df["label"].unique():
        sub_df = df[df["label"] == lbl]
        if len(sub_df) > 5:
            ax.hist(sub_df[feat], bins=50, alpha=0.5, label=lbl, density=True, range=(0, sub_df[feat].max() if feat != "dedx" else 0.1))
    ax.set_title(titles[i])
    ax.legend()

plt.tight_layout()
plt.show()

## 4. Train Classifiers

### 4a. Prepare Data Split

In [ ]:
feature_cols = [
    "length", "nhits", "z_depth", "straightness", "total_energy", "dedx",
    "start_z", "end_z", "dca", "angle", "dist_to_vtx", "length_rank", "z_depth_rank", "is_longest"
]
X = df[feature_cols]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set: {len(X_train)} tracks, Test set: {len(X_test)} tracks.")

### 4b. Baseline Rule-Based Cuts
We use a standard rule-based cut for muon selection: `length > 1800` mm AND `z_depth > 1500` mm.

In [ ]:
y_pred_baseline = np.where((X_test["length"] > 1800) & (X_test["z_depth"] > 1500), "muon", "other")
y_test_binary = np.where(y_test == "muon", "muon", "other")

cm_base = confusion_matrix(y_test_binary, y_pred_baseline, labels=["muon", "other"])
tn, fp, fn, tp = cm_base.ravel()

eff_base = tp / (tp + fn)
pur_base = tp / (tp + fp)
rej_base = tn / (tn + fp)

print("Baseline Confusion Matrix:")
print(cm_base)
print(f"Efficiency (Muon): {eff_base*100:.2f}%")
print(f"Purity (Muon):     {pur_base*100:.2f}%")
print(f"Hadron Rejection:  {rej_base*100:.2f}%")

### 4c. Machine Learning Classifier (Random Forest)

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

In [ ]:
classes = sorted(y.unique())
cm_rf = confusion_matrix(y_test, y_pred_rf, labels=classes)
cm_df = pd.DataFrame(cm_rf, index=classes, columns=classes)
print("Confusion Matrix:")
cm_df

### 4d. Random Forest Binary Evaluation
For direct comparison with baseline cuts, we evaluate the Random Forest in a binary classification mode (Muon vs Other).

In [ ]:
y_pred_rf_binary = np.where(y_pred_rf == "muon", "muon", "other")
cm_rf_bin = confusion_matrix(y_test_binary, y_pred_rf_binary, labels=["muon", "other"])
tn_rf, fp_rf, fn_rf, tp_rf = cm_rf_bin.ravel()

eff_rf = tp_rf / (tp_rf + fn_rf)
pur_rf = tp_rf / (tp_rf + fp_rf)
rej_rf = tn_rf / (tn_rf + fp_rf)

print("Random Forest Binary Confusion Matrix:")
print(cm_rf_bin)
print(f"RF Efficiency (Muon): {eff_rf*100:.2f}%")
print(f"RF Purity (Muon):     {pur_rf*100:.2f}%")
print(f"RF Hadron Rejection:  {rej_rf*100:.2f}%")

## 5. ROC Curve & Feature Importance

In [ ]:
# Plot feature importances
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(10, 6))
plt.bar(range(X.shape[1]), importances[indices], align="center", color='royalblue')
plt.xticks(range(X.shape[1]), [feature_cols[i] for i in indices], rotation=45, ha='right')
plt.title("Feature Importance (Random Forest)")
plt.ylabel("Importance Score")
plt.tight_layout()
plt.show()

## 6. Summary and Answers to Diagnostic Questions

1. **Can matched hits produce reasonable x, y, z?**
   Only partially (X and Z are resolved, Y collapses to zero due to small stereo angle amplification and bar segmentation limits).
2. **Which stereo convention matches Reco_Tree.TrackHitPos better?**
   Convention A ($dx \approx 252$ mm vs $521$ mm for Convention B).
3. **How large are the residuals?**
   - $dx \approx 252$ mm
   - $dy \approx 11,600$ mm (dominated by discretization effects and upstream extrapolation)
   - $dz \approx -40$ mm
4. **Is the reconstruction stable?**
   Highly unstable for single hits, requiring multi-plane track-level fits to globally determine slopes and intercepts.